In [1]:
import pandas as pd

path = "CRM+Sales+Opportunities/"   # لو الـ notebook جوه الفولدر نفسه خليها ""

accounts = pd.read_csv(path + "accounts.csv")
products = pd.read_csv(path + "products.csv")
teams = pd.read_csv(path + "sales_teams.csv")
pipeline = pd.read_csv(path + "sales_pipeline.csv")
dictionary = pd.read_csv(path + "data_dictionary.csv")

for name, df in [("accounts", accounts), ("products", products),
                 ("teams", teams), ("pipeline", pipeline)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

accounts: 85 rows, 7 columns
products: 7 rows, 3 columns
teams: 35 rows, 3 columns
pipeline: 8800 rows, 8 columns


In [1]:
# لو مش متسطّبة: pip install openpyxl
import pandas as pd
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

path = "CRM+Sales+Opportunities/"   # نفس الـ path اللي استخدمناه قبل كده
out = "CRM_Sales_Opportunities.xlsx"

files = {
    "accounts": "accounts.csv",
    "products": "products.csv",
    "sales_teams": "sales_teams.csv",
    "sales_pipeline": "sales_pipeline.csv",
    "data_dictionary": "data_dictionary.csv",
}

with pd.ExcelWriter(out, engine="openpyxl", datetime_format="yyyy-mm-dd") as writer:
    for sheet, f in files.items():
        df = pd.read_csv(path + f)

        # التواريخ لازم تتحول لـ datetime عشان Excel يعرفها كتواريخ
        if sheet == "sales_pipeline":
            df["engage_date"] = pd.to_datetime(df["engage_date"])
            df["close_date"] = pd.to_datetime(df["close_date"])

        df.to_excel(writer, sheet_name=sheet, index=False)
        ws = writer.sheets[sheet]

        # حوّل الـ range لـ Excel Table باسم الـ sheet
        ref = f"A1:{get_column_letter(df.shape[1])}{df.shape[0] + 1}"
        tbl = Table(displayName=sheet, ref=ref)
        tbl.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
        ws.add_table(tbl)

        # تثبيت الصف الأول وضبط عرض الأعمدة
        ws.freeze_panes = "A2"
        for i, col in enumerate(df.columns, 1):
            longest = max(df[col].astype(str).str.len().max(), len(col))
            ws.column_dimensions[get_column_letter(i)].width = min(longest + 2, 40)

print("تم:", out)

تم: CRM_Sales_Opportunities.xlsx


In [2]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
dictionary

,Table,Field,Description
0,accounts,account,Company name
1,accounts,sector,Industry
2,accounts,year_established,Year Established
3,accounts,revenue,Annual revenue (in millions of USD)
4,accounts,employees,Number of employees
5,accounts,office_location,Headquarters
6,accounts,subsidiary_of,Parent company
7,products,product,Product name
8,products,series,Product series
9,products,sales_price,Suggested retail price


In [3]:
print(pipeline.head())
print()
print(pipeline["deal_stage"].value_counts())
print()
print("Nulls:")
print(pipeline.isnull().sum())
print()
print(pipeline[["engage_date", "close_date"]].agg(["min", "max"]))
print()
print(pipeline["close_value"].describe())

  opportunity_id      sales_agent         product  account deal_stage  \
0       1C1I7A6R      Moses Frase  GTX Plus Basic  Cancity        Won   
1       Z063OYW0  Darcel Schlecht          GTXPro    Isdom        Won   
2       EC4QE1BX  Darcel Schlecht      MG Special  Cancity        Won   
3       MV1LWRNH      Moses Frase       GTX Basic  Codehow        Won   
4       PE84CX4O        Zane Levy       GTX Basic   Hatfan        Won   

  engage_date  close_date  close_value  
0  2016-10-20  2017-03-01       1054.0  
1  2016-10-25  2017-03-11       4514.0  
2  2016-10-25  2017-03-07         50.0  
3  2016-10-25  2017-03-09        588.0  
4  2016-10-25  2017-03-02        517.0  

deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64

Nulls:
opportunity_id       0
sales_agent          0
product              0
account           1425
deal_stage           0
engage_date        500
close_date        2089
close_value       2089
dtype

In [4]:
print("accounts مش موجودة في accounts:",
      (~pipeline["account"].dropna().isin(accounts["account"])).sum())
print("products مش موجودة في products:",
      (~pipeline["product"].isin(products["product"])).sum())
print("agents مش موجودين في teams:",
      (~pipeline["sales_agent"].isin(teams["sales_agent"])).sum())

accounts مش موجودة في accounts: 0
products مش موجودة في products: 1480
agents مش موجودين في teams: 0


In [5]:
bad = pipeline[~pipeline["product"].isin(products["product"])]
print(bad["product"].value_counts())
print()
print(products["product"].tolist())

product
GTXPro    1480
Name: count, dtype: int64

['GTX Basic', 'GTX Pro', 'MG Special', 'MG Advanced', 'GTX Plus Pro', 'GTX Plus Basic', 'GTK 500']


In [6]:
print(pipeline[pipeline["account"].isnull()]["deal_stage"].value_counts())
print()
print(pipeline.groupby("deal_stage")[
      "close_value"].agg(["count", "mean", "sum"]))

deal_stage
Engaging       1088
Prospecting     337
Name: count, dtype: int64

             count         mean         sum
deal_stage                                 
Engaging         0          NaN         0.0
Lost          2473     0.000000         0.0
Prospecting      0          NaN         0.0
Won           4238  2360.909391  10005534.0


In [7]:
print("=== accounts ===")
print(accounts.isnull().sum())
print(accounts["sector"].value_counts())
print()
print("=== products ===")
print(products)
print()
print("=== teams ===")
print(teams.groupby(["regional_office", "manager"]).size())
print()
print("=== checks على الـ pipeline ===")
print("opportunity_id مكرر:", pipeline["opportunity_id"].duplicated().sum())
print("close قبل engage:",
      (pipeline["close_date"] < pipeline["engage_date"]).sum())
print("Won بقيمة صفر:", ((pipeline["deal_stage"] == "Won") & (
    pipeline["close_value"] == 0)).sum())

=== accounts ===
account              0
sector               0
year_established     0
revenue              0
employees            0
office_location      0
subsidiary_of       70
dtype: int64
sector
retail                17
technolgy             12
medical               12
marketing              8
finance                8
software               7
entertainment          6
telecommunications     6
services               5
employment             4
Name: count, dtype: int64

=== products ===
          product series  sales_price
0       GTX Basic    GTX          550
1         GTX Pro    GTX         4821
2      MG Special     MG           55
3     MG Advanced     MG         3393
4    GTX Plus Pro    GTX         5482
5  GTX Plus Basic    GTX         1096
6         GTK 500    GTK        26768

=== teams ===
regional_office  manager         
Central          Dustin Brinkmann    5
                 Melvin Marxen       6
East             Cara Losch          6
                 Rocco Neubert       6